سلول ۱ — مسیرها و خواندن فایل‌ها

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(".")

POS_DIR = PROJECT_ROOT / "Data_proc" / "positives"
NEG_DIR = PROJECT_ROOT / "Data_proc" / "negatives"
PAIR_DIR = PROJECT_ROOT / "Data_proc" / "pairs"
QC_DIR = PROJECT_ROOT / "Data_proc" / "qc_reports"

for d in [PAIR_DIR, QC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

POS_ALL_PATH = POS_DIR / "positive_all.csv"
NEG_ALL_PATH = NEG_DIR / "negative_all_after_ppi_loc_scrna_filter.csv"

print("Positive file exists:", POS_ALL_PATH.exists(), POS_ALL_PATH)
print("Negative file exists:", NEG_ALL_PATH.exists(), NEG_ALL_PATH)

pos = pd.read_csv(POS_ALL_PATH, dtype=str, low_memory=False)
neg = pd.read_csv(NEG_ALL_PATH, dtype=str, low_memory=False)

pos["label"] = pos["label"].astype(int)
neg["label"] = neg["label"].astype(int)

print("Positive:", pos.shape)
print("Negative:", neg.shape)

display(pos.head())
display(neg.head())

سلول ۲ — استانداردسازی ستون‌ها و بازسازی شناسه‌ها

In [ ]:
BASE_COLS = [
    "pair_id",
    "group_id",
    "enzyme_class",
    "enz_ac",
    "sub_ac",
    "enz_gene",
    "sub_gene",
    "enzyme_type",
    "label",
    "source",
    "pmid",
]

def normalize_ac(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none", "null", "-", "na", "n/a"}:
        return np.nan
    x = x.split(";")[0].split(",")[0].split("|")[0].strip()
    x = x.split("-")[0].strip()
    return x

def normalize_gene(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none", "null", "-", "na", "n/a"}:
        return np.nan
    return x

def standardize_pair_df(df, label_value, name):
    out = df.copy()
    
    # Ensure required base columns exist
    for c in BASE_COLS:
        if c not in out.columns:
            out[c] = np.nan
    
    out["enzyme_class"] = out["enzyme_class"].astype(str).str.strip()
    out["enz_ac"] = out["enz_ac"].map(normalize_ac)
    out["sub_ac"] = out["sub_ac"].map(normalize_ac)
    out["enz_gene"] = out["enz_gene"].map(normalize_gene)
    out["sub_gene"] = out["sub_gene"].map(normalize_gene)
    out["label"] = int(label_value)
    
    out["pair_id"] = (
        out["enzyme_class"].astype(str)
        + "|"
        + out["enz_ac"].astype(str)
        + "|"
        + out["sub_ac"].astype(str)
    )
    
    out["group_id"] = (
        out["enzyme_class"].astype(str)
        + "|"
        + out["enz_ac"].astype(str)
    )
    
    out = out.dropna(subset=["enzyme_class", "enz_ac", "sub_ac"]).copy()
    
    # Keep base cols first, then any extra annotation columns
    extra_cols = [c for c in out.columns if c not in BASE_COLS]
    out = out[BASE_COLS + extra_cols]
    
    print(f"{name}: {df.shape} -> {out.shape}")
    return out

pos_final = standardize_pair_df(pos, label_value=1, name="positive")
neg_final = standardize_pair_df(neg, label_value=0, name="negative")

display(pos_final.head())
display(neg_final.head())

سلول ۳ — merge و conflict check

In [ ]:
pairs_all = pd.concat([pos_final, neg_final], ignore_index=True)

# Duplicate exact pair_id
dup_pair_ids = pairs_all[
    pairs_all.duplicated("pair_id", keep=False)
].sort_values("pair_id")

# Label conflict: same pair_id with both 0 and 1
label_conflict_keys = (
    pairs_all.groupby("pair_id")["label"]
    .nunique()
    .reset_index(name="n_labels")
)
label_conflict_keys = label_conflict_keys[label_conflict_keys["n_labels"] > 1]

label_conflicts = pairs_all[
    pairs_all["pair_id"].isin(label_conflict_keys["pair_id"])
].sort_values("pair_id")

print("All pairs:", pairs_all.shape)
print("Unique pair_id:", pairs_all["pair_id"].nunique())
print("Duplicate pair_id rows:", len(dup_pair_ids))
print("Label conflict pair_ids:", len(label_conflict_keys))

print("\nLabel counts:")
print(pairs_all["label"].value_counts(dropna=False))

print("\nEnzyme class counts:")
print(pairs_all["enzyme_class"].value_counts(dropna=False))

display(dup_pair_ids.head(20))
display(label_conflicts.head(20))

سلول ۴ — QC نهایی pair table

In [ ]:
def qc_pair_table(df, name):
    return {
        "dataset": name,
        "n_rows": len(df),
        "n_unique_pair_id": df["pair_id"].nunique(),
        "n_duplicate_pair_id_rows": int(df.duplicated("pair_id").sum()),
        "n_unique_group_id": df["group_id"].nunique(),
        "n_unique_enz_ac": df["enz_ac"].nunique(),
        "n_unique_sub_ac": df["sub_ac"].nunique(),
        "n_missing_enzyme_class": int(df["enzyme_class"].isna().sum()),
        "n_missing_enz_ac": int(df["enz_ac"].isna().sum()),
        "n_missing_sub_ac": int(df["sub_ac"].isna().sum()),
        "n_missing_enz_gene": int(df["enz_gene"].isna().sum()),
        "n_missing_sub_gene": int(df["sub_gene"].isna().sum()),
        "n_pair_id_starts_with_nan": int(df["pair_id"].astype(str).str.startswith("nan|").sum()),
        "n_label_0": int((df["label"] == 0).sum()),
        "n_label_1": int((df["label"] == 1).sum()),
        "n_E3": int((df["enzyme_class"] == "E3").sum()),
        "n_DUB": int((df["enzyme_class"] == "DUB").sum()),
    }

final_pairs_qc = pd.DataFrame([
    qc_pair_table(pos_final, "positive_final"),
    qc_pair_table(neg_final, "negative_final"),
    qc_pair_table(pairs_all, "pairs_all_final"),
])

display(final_pairs_qc)

print("Class balance:")
print(pairs_all["label"].value_counts(normalize=True))

سلول ۵ — accession lists برای مرحله FASTA و embedding

In [ ]:
required_enzymes = (
    pairs_all[["enz_ac"]]
    .dropna()
    .drop_duplicates()
    .sort_values("enz_ac")
    .reset_index(drop=True)
)

required_substrates = (
    pairs_all[["sub_ac"]]
    .dropna()
    .drop_duplicates()
    .sort_values("sub_ac")
    .reset_index(drop=True)
)

required_accessions = pd.DataFrame({
    "accession": sorted(
        set(required_enzymes["enz_ac"].astype(str)) |
        set(required_substrates["sub_ac"].astype(str))
    )
})

print("Required enzymes:", required_enzymes.shape)
print("Required substrates:", required_substrates.shape)
print("Required all accessions:", required_accessions.shape)

display(required_accessions.head())

سلول ۶ — ذخیره خروجی‌ها

In [ ]:
pos_final.to_csv(
    PAIR_DIR / "pairs_positive_final.csv",
    index=False
)

neg_final.to_csv(
    PAIR_DIR / "pairs_negative_final.csv",
    index=False
)

pairs_all.to_csv(
    PAIR_DIR / "pairs_all_final.csv",
    index=False
)

required_enzymes.to_csv(
    PAIR_DIR / "required_enzymes_final.csv",
    index=False
)

required_substrates.to_csv(
    PAIR_DIR / "required_substrates_final.csv",
    index=False
)

required_accessions.to_csv(
    PAIR_DIR / "required_accessions_final.csv",
    index=False
)

final_pairs_qc.to_csv(
    QC_DIR / "final_pairs_qc.csv",
    index=False
)

dup_pair_ids.to_csv(
    QC_DIR / "final_pairs_duplicate_pair_ids.csv",
    index=False
)

label_conflicts.to_csv(
    QC_DIR / "final_pairs_label_conflicts.csv",
    index=False
)

print("Saved:")
print(PAIR_DIR / "pairs_positive_final.csv")
print(PAIR_DIR / "pairs_negative_final.csv")
print(PAIR_DIR / "pairs_all_final.csv")
print(PAIR_DIR / "required_accessions_final.csv")
print(QC_DIR / "final_pairs_qc.csv")

سلول ۷ — چک فایل‌های ذخیره‌شده

In [ ]:
check_files = [
    PAIR_DIR / "pairs_positive_final.csv",
    PAIR_DIR / "pairs_negative_final.csv",
    PAIR_DIR / "pairs_all_final.csv",
    PAIR_DIR / "required_enzymes_final.csv",
    PAIR_DIR / "required_substrates_final.csv",
    PAIR_DIR / "required_accessions_final.csv",
    QC_DIR / "final_pairs_qc.csv",
    QC_DIR / "final_pairs_duplicate_pair_ids.csv",
    QC_DIR / "final_pairs_label_conflicts.csv",
]

for f in check_files:
    print(f.name, "exists:", f.exists())
    if f.exists() and f.suffix == ".csv":
        tmp = pd.read_csv(f, dtype=str, low_memory=False)
        print("shape:", tmp.shape)